In [2]:
# 1. pip install
%pip install pandas numpy requests jupyter python-dotenv

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: torchsde 0.2.5 has a non-standard dependency specifier numpy>=1.19.*; python_version >= "3.7". pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of torchsde or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# 🚧 서울시 도로굴착 공사현황 데이터 전처리
# FR-005: 음성 안내 서비스 - 데이터 전처리 단계

import pandas as pd
import numpy as np
import re
import requests
import json
from datetime import datetime, timedelta
import time
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# 환경변수 로드
load_dotenv()


True

In [7]:

print("🔧 도로굴착 공사현황 데이터 전처리 시작")
print("🔐 환경변수 로드 완료")
print("=" * 50)

print("🔧 도로굴착 공사현황 데이터 전처리 시작")
print("=" * 50)

# =============================================================================
# 1. 데이터 로드 및 기본 정보 확인
# =============================================================================

def load_road_excavation_data(file_path="../../data/csv/서울시 도로굴착 공사 현황.csv"):
    """
    도로굴착 현황 CSV 파일 로드
    
    Args:
        file_path (str): CSV 파일 경로
    
    Returns:
        pd.DataFrame: 로드된 데이터프레임
    """
    try:
        # 여러 인코딩으로 시도
        encodings = ['cp949', 'euc-kr', 'utf-8']
        
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding)
                print(f"✅ 파일 로드 성공 (인코딩: {encoding})")
                break
            except UnicodeDecodeError:
                continue
        else:
            raise Exception("모든 인코딩 시도 실패")
            
        # 기본 정보 출력
        print(f"📊 데이터 개수: {len(df):,}건")
        print(f"📋 컬럼 개수: {len(df.columns)}개")
        print(f"📅 데이터 미리보기:")
        print("-" * 50)
        
        return df
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        print("💡 data/csv/ 폴더에 '서울시 도로굴착 공사 현황.csv' 파일이 있는지 확인하세요.")
        return None
    except Exception as e:
        print(f"❌ 파일 로드 중 오류 발생: {str(e)}")
        return None

# 데이터 로드
df_raw = load_road_excavation_data()

if df_raw is not None:
    # 컬럼명 확인
    print("\n🔍 컬럼 정보:")
    for i, col in enumerate(df_raw.columns):
        print(f"{i+1:2d}. {col}")
    
    # 데이터 미리보기
    print("\n📋 데이터 샘플 (첫 3행):")
    print(df_raw.head(3).to_string())
    
    # 기본 통계
    print(f"\n📈 기본 통계:")
    print(f"- 총 데이터 수: {len(df_raw):,}건")
    print(f"- 자치구별 분포:")
    if '구' in df_raw.columns:
        print(df_raw['구'].value_counts().head(10))


🔧 도로굴착 공사현황 데이터 전처리 시작
🔐 환경변수 로드 완료
🔧 도로굴착 공사현황 데이터 전처리 시작
✅ 파일 로드 성공 (인코딩: cp949)
📊 데이터 개수: 533건
📋 컬럼 개수: 10개
📅 데이터 미리보기:
--------------------------------------------------

🔍 컬럼 정보:
 1. 관리코드
 2. 구
 3. 동
 4. 공사명
 5. 착공일 ~ 준공일
 6. 신청자
 7. 처리상태
 8. 도로
 9. 포장
10. 허가번호

📋 데이터 샘플 (첫 3행):
           관리코드    구    동                                                                             공사명                착공일 ~ 준공일            신청자    처리상태  도로  포장               허가번호
0  일반-210728-75  서초구  잠원동    잠원동 신반포14차조합(롯데건설)3000kW 신설_보도(대선, 송영아)<BR>(서초구 잠원동   잠원동112 ~ 잠원동   잠원동112)  2022-06-20 ~ 2022-08-31    한국전력공사 서초지사  착공계 접수  구도  보도  서초구-2021-전기-00126
1  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(용산구 동자동   35-41 ~ 동자동   35-41부근)  2022-06-16 ~ 2022-08-14  한국전력공사 마포용산지사  착공계 접수  구도  차도  용산구-2021-전기-00079
2  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(용산구 동자동   19-87 ~ 동자동   19-87부근)  2022-06-16 ~ 2022-08-14  한국전력공사 마포용산지사  착공계 접수  구도  차도  용산구-2021-

In [8]:

# =============================================================================
# 2. 텍스트 정제 함수
# =============================================================================

def clean_construction_text(df):
    """
    공사명 및 관련 텍스트 필드 정제
    
    Args:
        df (pd.DataFrame): 원본 데이터프레임
    
    Returns:
        pd.DataFrame: 정제된 데이터프레임
    """
    df_clean = df.copy()
    
    print("🧹 텍스트 정제 시작...")
    
    # 1. HTML 태그 제거
    if '공사명' in df_clean.columns:
        df_clean['공사명_정제'] = df_clean['공사명'].str.replace('<BR>', ' ', regex=False)
        df_clean['공사명_정제'] = df_clean['공사명_정제'].str.replace('<br>', ' ', regex=False)
        print("✅ HTML 태그 제거 완료")
    
    # 2. 괄호 안 주소 정보 추출
    # 패턴: "(구 동 주소)" 형태 추출
    if '공사명_정제' in df_clean.columns:
        # 정규식으로 주소 추출 - 구와 동이 포함된 패턴
        address_pattern = r'\(([^)]*구[^)]*동[^)]*)\)'
        df_clean['추출주소'] = df_clean['공사명_정제'].str.extract(address_pattern)
        
        # 더 구체적인 주소 패턴도 시도
        detailed_pattern = r'\(([^)]*구 [^)]*)\)'
        mask = df_clean['추출주소'].isna()
        df_clean.loc[mask, '추출주소'] = df_clean.loc[mask, '공사명_정제'].str.extract(detailed_pattern)
        
        print("✅ 주소 정보 추출 완료")
        print(f"   - 추출된 주소 개수: {df_clean['추출주소'].notna().sum():,}건")
    
    # 3. 공사명에서 불필요한 정보 제거
    if '공사명_정제' in df_clean.columns:
        df_clean['공사명_단순'] = df_clean['공사명_정제'].str.split('(').str[0].str.strip()
        print("✅ 공사명 단순화 완료")
    
    # 4. 공사 종류 추출 (키워드 기반)
    construction_types = {
        '전력': ['전력', '전주', '케이블', '한국전력'],
        '통신': ['통신', '인터넷', 'KT', 'SK', 'LG'],
        '가스': ['가스', '도시가스'],
        '상하수도': ['상수도', '하수도', '하수관', '급수', '배수'],
        '도로': ['도로', '아스팔트', '포장'],
        '지하철': ['지하철', '전철', '도시철도'],
        '건물': ['건설', '신축', '증축', '재건축']
    }
    
    df_clean['공사종류'] = '기타'
    
    for category, keywords in construction_types.items():
        mask = df_clean['공사명_단순'].str.contains('|'.join(keywords), case=False, na=False)
        df_clean.loc[mask, '공사종류'] = category
    
    print("✅ 공사 종류 분류 완료")
    print("   - 공사종류별 분포:")
    print(df_clean['공사종류'].value_counts())
    
    return df_clean

# 텍스트 정제 실행
if df_raw is not None:
    df_cleaned = clean_construction_text(df_raw)
    print(f"\n📊 정제 후 컬럼 수: {len(df_cleaned.columns)}개")


🧹 텍스트 정제 시작...
✅ HTML 태그 제거 완료
✅ 주소 정보 추출 완료
   - 추출된 주소 개수: 533건
✅ 공사명 단순화 완료
✅ 공사 종류 분류 완료
   - 공사종류별 분포:
공사종류
기타      371
지하철      47
상하수도     46
건물       32
통신       24
전력       10
가스        3
Name: count, dtype: int64

📊 정제 후 컬럼 수: 14개


In [9]:

# =============================================================================
# 3. 날짜 파싱 함수
# =============================================================================

def parse_construction_dates(df):
    """
    공사 기간 정보 파싱
    
    Args:
        df (pd.DataFrame): 정제된 데이터프레임
    
    Returns:
        pd.DataFrame: 날짜 파싱된 데이터프레임
    """
    df_dated = df.copy()
    
    print("📅 날짜 정보 파싱 시작...")
    
    if '착공일 ~ 준공일' in df_dated.columns:
        # 날짜 문자열 분할
        date_parts = df_dated['착공일 ~ 준공일'].str.split(' ~ ', expand=True)
        
        # 시작일과 종료일 파싱
        df_dated['시작일'] = pd.to_datetime(date_parts[0], errors='coerce')
        df_dated['종료일'] = pd.to_datetime(date_parts[1], errors='coerce')
        
        # 공사 기간 계산 (일 단위)
        df_dated['공사기간_일'] = (df_dated['종료일'] - df_dated['시작일']).dt.days
        
        # 현재 날짜 기준 상태 계산
        today = datetime.now()
        df_dated['현재날짜'] = today
        
        # 공사 상태 분류
        conditions = [
            df_dated['시작일'] > today,  # 미착공
            (df_dated['시작일'] <= today) & (df_dated['종료일'] >= today),  # 진행중
            df_dated['종료일'] < today   # 완료
        ]
        choices = ['미착공', '진행중', '완료']
        df_dated['공사상태_계산'] = np.select(conditions, choices, default='불명')
        
        # 진행율 계산 (진행중인 공사만)
        mask_ongoing = df_dated['공사상태_계산'] == '진행중'
        if mask_ongoing.sum() > 0:
            elapsed_days = (today - df_dated.loc[mask_ongoing, '시작일']).dt.days
            total_days = df_dated.loc[mask_ongoing, '공사기간_일']
            df_dated.loc[mask_ongoing, '진행율'] = (elapsed_days / total_days * 100).clip(0, 100)
        
        print("✅ 날짜 파싱 완료")
        print(f"   - 유효한 시작일: {df_dated['시작일'].notna().sum():,}건")
        print(f"   - 유효한 종료일: {df_dated['종료일'].notna().sum():,}건")
        print("   - 공사 상태별 분포:")
        print(df_dated['공사상태_계산'].value_counts())
        
        # 날짜 범위 확인
        if df_dated['시작일'].notna().sum() > 0:
            print(f"   - 최초 시작일: {df_dated['시작일'].min()}")
            print(f"   - 최종 종료일: {df_dated['종료일'].max()}")
    
    return df_dated

# 날짜 파싱 실행
if 'df_cleaned' in locals():
    df_with_dates = parse_construction_dates(df_cleaned)


📅 날짜 정보 파싱 시작...
✅ 날짜 파싱 완료
   - 유효한 시작일: 372건
   - 유효한 종료일: 372건
   - 공사 상태별 분포:
공사상태_계산
완료     338
불명     161
진행중     34
Name: count, dtype: int64
   - 최초 시작일: 2020-08-27 00:00:00
   - 최종 종료일: 2026-07-14 00:00:00


In [12]:

# =============================================================================
# 4. 카카오맵 API 연동 함수 (환경변수 사용)
# =============================================================================

class KakaoGeocodingService:
    """카카오맵 API를 사용한 지오코딩 서비스"""
    
    def __init__(self):
        self.api_key = os.getenv('KAKAO_REST_API_KEY')
        self.base_url = "https://dapi.kakao.com/v2/local/search/address.json"
        self.delay = float(os.getenv('GEOCODING_DELAY', 0.1))
        self.max_calls_per_minute = int(os.getenv('MAX_API_CALLS_PER_MINUTE', 100))
        self.batch_size = int(os.getenv('BATCH_SIZE', 50))
        
        # 서울시 좌표 범위
        self.seoul_bounds = {
            'lat_min': float(os.getenv('SEOUL_LAT_MIN', 37.4)),
            'lat_max': float(os.getenv('SEOUL_LAT_MAX', 37.7)),
            'lng_min': float(os.getenv('SEOUL_LNG_MIN', 126.8)),
            'lng_max': float(os.getenv('SEOUL_LNG_MAX', 127.2))
        }
        
        if not self.api_key:
            print("⚠️  KAKAO_REST_API_KEY가 설정되지 않았습니다.")
            print("💡 .env 파일에 API 키를 설정하거나 더미 데이터로 진행합니다.")
    
    def get_coordinates(self, address):
        """
        단일 주소를 좌표로 변환
        
        Args:
            address (str): 변환할 주소
        
        Returns:
            dict: {'lat': 위도, 'lng': 경도, 'status': 상태} 또는 None
        """
        if pd.isna(address) or address.strip() == '':
            return {'lat': None, 'lng': None, 'status': '빈주소'}
        
        if not self.api_key:
            return None
        
        headers = {"Authorization": f"KakaoAK {self.api_key}"}
        params = {"query": address}
        
        try:
            time.sleep(self.delay)  # Rate Limit 방지
            response = requests.get(self.base_url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data['documents']:
                    result = data['documents'][0]
                    lat, lng = float(result['y']), float(result['x'])
                    
                    # 서울시 범위 검증
                    if self._is_in_seoul(lat, lng):
                        return {'lat': lat, 'lng': lng, 'status': '성공'}
                    else:
                        return {'lat': lat, 'lng': lng, 'status': '범위외'}
                else:
                    return {'lat': None, 'lng': None, 'status': '결과없음'}
            elif response.status_code == 429:
                print("⚠️  API 호출 한도 초과. 잠시 대기...")
                time.sleep(60)  # 1분 대기
                return self.get_coordinates(address)  # 재시도
            else:
                return {'lat': None, 'lng': None, 'status': f'API오류({response.status_code})'}
                
        except requests.exceptions.Timeout:
            return {'lat': None, 'lng': None, 'status': '타임아웃'}
        except Exception as e:
            return {'lat': None, 'lng': None, 'status': f'오류({str(e)[:20]})'}
    
    def _is_in_seoul(self, lat, lng):
        """서울시 좌표 범위 내 여부 확인"""
        return (self.seoul_bounds['lat_min'] <= lat <= self.seoul_bounds['lat_max'] and
                self.seoul_bounds['lng_min'] <= lng <= self.seoul_bounds['lng_max'])
    
    def generate_dummy_coordinates(self, count):
        """서울시 범위 내 더미 좌표 생성"""
        lat_coords = np.random.uniform(
            self.seoul_bounds['lat_min'], 
            self.seoul_bounds['lat_max'], 
            count
        )
        lng_coords = np.random.uniform(
            self.seoul_bounds['lng_min'], 
            self.seoul_bounds['lng_max'], 
            count
        )
        return lat_coords, lng_coords

def convert_addresses_to_coordinates(df):
    """
    데이터프레임의 주소들을 좌표로 변환 (환경변수 기반)
    
    Args:
        df (pd.DataFrame): 주소가 포함된 데이터프레임
    
    Returns:
        pd.DataFrame: 좌표가 추가된 데이터프레임
    """
    df_coord = df.copy()
    geocoder = KakaoGeocodingService()
    
    print("🗺️ 주소 → 좌표 변환 시작...")
    
    if not geocoder.api_key:
        print("📝 API 키 없음 → 더미 좌표 생성")
        
        # 더미 좌표 생성
        lat_coords, lng_coords = geocoder.generate_dummy_coordinates(len(df_coord))
        df_coord['위도'] = lat_coords
        df_coord['경도'] = lng_coords
        df_coord['좌표변환상태'] = '더미데이터'
        df_coord['변환사용주소'] = '더미주소'
        
        print(f"✅ 더미 좌표 생성 완료: {len(df_coord)}건")
        return df_coord
    
    # 주소 우선순위 설정 및 변환 대상 준비
    addresses_to_convert = []
    
    for idx, row in df_coord.iterrows():
        if pd.notna(row.get('추출주소')) and row.get('추출주소').strip():
            # 추출된 주소 사용 (더 구체적)
            address = f"서울시 {row['추출주소']}"
        else:
            # 기본 주소 사용
            address = f"서울시 {row['구']} {row['동']}"
        
        addresses_to_convert.append((idx, address))
    
    print(f"📍 변환할 주소 수: {len(addresses_to_convert):,}개")
    print(f"🔄 배치 크기: {geocoder.batch_size}개씩 처리")
    
    # 배치 단위로 좌표 변환 실행
    all_results = []
    failed_count = 0
    
    for batch_start in range(0, len(addresses_to_convert), geocoder.batch_size):
        batch_end = min(batch_start + geocoder.batch_size, len(addresses_to_convert))
        batch = addresses_to_convert[batch_start:batch_end]
        
        print(f"   배치 {batch_start//geocoder.batch_size + 1}: "
              f"{batch_start+1}-{batch_end} ({len(batch)}개)")
        
        batch_results = []
        for idx, address in batch:
            result = geocoder.get_coordinates(address)
            
            if result and result['lat'] is not None:
                batch_results.append({
                    'index': idx,
                    'lat': result['lat'],
                    'lng': result['lng'],
                    'status': result['status'],
                    'address_used': address
                })
            else:
                failed_count += 1
                status = result['status'] if result else '알수없음'
                batch_results.append({
                    'index': idx,
                    'lat': None,
                    'lng': None,
                    'status': status,
                    'address_used': address
                })
        
        all_results.extend(batch_results)
        
        # 배치 간 대기 (Rate Limit 방지)
        if batch_end < len(addresses_to_convert):
            time.sleep(1)
    
    # 결과를 데이터프레임에 반영
    coord_df = pd.DataFrame(all_results)
    df_coord = df_coord.merge(coord_df, left_index=True, right_on='index', how='left')
    
    # 컬럼명 정리
    df_coord.rename(columns={
        'lat': '위도', 
        'lng': '경도', 
        'status': '좌표변환상태',
        'address_used': '변환사용주소'
    }, inplace=True)
    df_coord.drop('index', axis=1, inplace=True)
    
    # 결과 요약
    success_count = len(all_results) - failed_count
    success_rate = (success_count / len(all_results) * 100) if all_results else 0
    
    print("✅ 좌표 변환 완료")
    print(f"   - 성공: {success_count:,}건")
    print(f"   - 실패: {failed_count:,}건")
    print(f"   - 성공률: {success_rate:.1f}%")
    
    # 상태별 분포
    if '좌표변환상태' in df_coord.columns:
        print("   - 상태별 분포:")
        status_counts = df_coord['좌표변환상태'].value_counts()
        for status, count in status_counts.items():
            print(f"     * {status}: {count:,}건")
    
    return df_coord

# API 키 설정 (환경변수에서 로드)
print("🔑 환경변수 확인:")
print(f"   - KAKAO_REST_API_KEY: {'✅ 설정됨' if os.getenv('KAKAO_REST_API_KEY') else '❌ 없음'}")
print(f"   - GEOCODING_DELAY: {os.getenv('GEOCODING_DELAY', '기본값 사용')}")
print(f"   - BATCH_SIZE: {os.getenv('BATCH_SIZE', '기본값 사용')}")

# 좌표 변환 실행
if 'df_with_dates' in locals():
    df_with_coords = convert_addresses_to_coordinates(df_with_dates)


🔑 환경변수 확인:
   - KAKAO_REST_API_KEY: ✅ 설정됨
   - GEOCODING_DELAY: 0.1
   - BATCH_SIZE: 50
🗺️ 주소 → 좌표 변환 시작...
📍 변환할 주소 수: 533개
🔄 배치 크기: 50개씩 처리
   배치 1: 1-50 (50개)
   배치 2: 51-100 (50개)
   배치 3: 101-150 (50개)
   배치 4: 151-200 (50개)
   배치 5: 201-250 (50개)
   배치 6: 251-300 (50개)
   배치 7: 301-350 (50개)
   배치 8: 351-400 (50개)
   배치 9: 401-450 (50개)
   배치 10: 451-500 (50개)
   배치 11: 501-533 (33개)
✅ 좌표 변환 완료
   - 성공: 0건
   - 실패: 533건
   - 성공률: 0.0%
   - 상태별 분포:
     * API오류(403): 533건


In [ ]:

# =============================================================================
# 5. 최종 데이터 검증 및 요약
# =============================================================================

def validate_and_summarize(df):
    """
    전처리된 데이터 검증 및 요약
    
    Args:
        df (pd.DataFrame): 전처리된 데이터프레임
    
    Returns:
        dict: 검증 결과 및 요약 정보
    """
    print("🔍 데이터 검증 및 요약...")
    
    summary = {
        'total_records': len(df),
        'columns': list(df.columns),
        'data_quality': {},
        'statistics': {}
    }
    
    # 1. 데이터 품질 검증
    quality_checks = {
        '유효한_좌표': (df['위도'].notna() & df['경도'].notna()).sum(),
        '유효한_시작일': df['시작일'].notna().sum(),
        '유효한_종료일': df['종료일'].notna().sum(),
        '진행중_공사': (df['공사상태_계산'] == '진행중').sum(),
        '서울시_좌표범위': ((df['위도'] >= 37.4) & (df['위도'] <= 37.7) & 
                          (df['경도'] >= 126.8) & (df['경도'] <= 127.2)).sum()
    }
    
    summary['data_quality'] = quality_checks
    
    # 2. 통계 정보
    if df['공사기간_일'].notna().sum() > 0:
        summary['statistics']['평균_공사기간'] = df['공사기간_일'].mean()
        summary['statistics']['최대_공사기간'] = df['공사기간_일'].max()
        summary['statistics']['최소_공사기간'] = df['공사기간_일'].min()
    
    # 3. 분포 정보
    summary['distributions'] = {
        '자치구별': df['구'].value_counts().to_dict(),
        '공사종류별': df['공사종류'].value_counts().to_dict(),
        '공사상태별': df['공사상태_계산'].value_counts().to_dict()
    }
    
    # 결과 출력
    print("=" * 50)
    print("📊 최종 데이터 요약")
    print("=" * 50)
    print(f"총 레코드 수: {summary['total_records']:,}건")
    print(f"총 컬럼 수: {len(summary['columns'])}개")
    
    print("\n🎯 데이터 품질:")
    for key, value in quality_checks.items():
        percentage = (value / summary['total_records'] * 100) if summary['total_records'] > 0 else 0
        print(f"  - {key}: {value:,}건 ({percentage:.1f}%)")
    
    if summary.get('statistics'):
        print("\n📈 공사기간 통계:")
        for key, value in summary['statistics'].items():
            print(f"  - {key}: {value:.1f}일")
    
    print("\n🏢 자치구별 상위 5개:")
    for district, count in list(summary['distributions']['자치구별'].items())[:5]:
        print(f"  - {district}: {count:,}건")
    
    print("\n🔧 공사종류별 분포:")
    for const_type, count in summary['distributions']['공사종류별'].items():
        print(f"  - {const_type}: {count:,}건")
    
    return summary

# 최종 검증 실행
if 'df_with_coords' in locals():
    final_summary = validate_and_summarize(df_with_coords)
    
    # 최종 데이터 저장
    output_path = "../../data/processed/road_excavation_processed.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    try:
        df_with_coords.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n💾 처리된 데이터 저장 완료: {output_path}")
        print(f"   파일 크기: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")
    except Exception as e:
        print(f"❌ 파일 저장 실패: {str(e)}")

print("\n🎉 도로굴착 공사현황 데이터 전처리 완료!")
print("=" * 50)